In [9]:
# Proximity Funktion mit Gaussian-Peaks, getrennten f- und k-Strafen
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde


def proximity_max_freq_gaussian_fk(
    x,
    sharpness_base=10.0,     # s0: Grundschärfe
    sharpness_growth=2.0,    # β: Schärfe-Exponent auf k
    decay=0.5,               # α: Dämpfung auf k
    max_freq=4,
    weight_exponent=1.0,     # w: Dämpfung auf f
    freq_sharpness_exp=1.0,  # γ: Schärfe-Exponent auf f
    norm_x=1.0,
    output_max=1.0,
    clamp_radius_factor=0.5  # wie viel vom Abstand zum Nachbar-Peak erlaubt ist
):
    x = np.array(x)
    all_peak_values = []

    for f in range(1, max_freq + 1):
        # Gewichtung nach Frequenz f
        weight_f = 1 / (f ** weight_exponent)

        peaks_f = np.zeros_like(x, dtype=float)
        max_k = int(np.ceil(x.max() * f)) + 1

        for k in range(1, max_k + 1):
            mu = k / f

            # Diskrete Schärfe
            sharpness = sharpness_base * (f ** freq_sharpness_exp) * (k ** sharpness_growth)
            sigma = 1 / np.sqrt(sharpness + 1e-9)

            # Decay auf k
            weight_k = 1 / (k ** decay)

            # Cutoff-Radius: halber Abstand zum nächsten Peak
            r = clamp_radius_factor / f
            mask = (x >= mu - r) & (x <= mu + r)

            g = np.zeros_like(x, dtype=float)
            g[mask] = np.exp(-0.5 * ((x[mask] - mu) / sigma) ** 2)

            # Gesamtgewichtung f & k
            peaks_f = np.maximum(peaks_f, weight_f * weight_k * g)

        all_peak_values.append(peaks_f)

    result = np.maximum.reduce(all_peak_values)

    # Normierung bei norm_x
    max_at_norm_x = np.max([
        (1 / (f ** weight_exponent)) * (1 / (k ** decay))
        for f in range(1, max_freq + 1)
        for k in range(1, int(np.ceil(norm_x * f)) + 1)
    ])
    return result / max_at_norm_x * output_max



def plot_proximity_max_freq_gaussian_fk(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq_gaussian_fk(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = Mittelwert
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit Gaussian-Peaks (getrennte f- und k-Strafen)", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(x_vals, [mean_prox] * len(x_vals),
                 color='red', linestyle='--', linewidth=2,
                 label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit Gaussian-Peaks")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, bins_hist, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")

    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian_fk,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base (s0)"),
    sharpness_growth=FloatSlider(min=0, max=5, step=0.1, value=2, description="k-Schärfe (β)"),
    decay=FloatSlider(min=0, max=4, step=0.05, value=0.5, description="k-Decay (α)"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency (f)"),
    weight_exponent=FloatSlider(min=0, max=4, step=0.1, value=1, description="f-Gewichtung (w)"),
    freq_sharpness_exp=FloatSlider(min=0, max=4, step=0.1, value=1, description="f-Schärfe (γ)"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base (s0)', max=50.0, min=1.0, step=1.0),…

<function __main__.plot_proximity_max_freq_gaussian_fk(sharpness_base=10.0, sharpness_growth=2.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [7]:
# Proximity Funktion mit Gauß-Peaks (diskret pro Peak), exponentieller Decay
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde


def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,              # exponentielle Dämpfung
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_peak_values = []

    for freq in range(1, max_freq + 1):
        # Frequenzgewichtung
        weight = 1 / (freq ** weight_exponent)

        # exponentielle Dämpfung (diskret pro freq)
        decay_factor = np.exp(-decay * freq)

        # Maxima der Gauss-Kette
        gauss_max = np.zeros_like(x, dtype=float)
        max_k = int(np.ceil(x.max() * freq)) + 1

        for k in range(1, max_k + 1):
            mu = k / freq

            # diskrete Sharpness pro Peak
            freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
            freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)
            sharpness = freq_sharpness_base * (mu ** freq_sharpness_growth)

            sigma = 1 / np.sqrt(sharpness + 1e-9)

            # Peak = Gauß-Kurve (Höhe 1, Fläche ≠ 1)
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)

            # Dämpfung + Gewichtung pro Peak
            gauss_max = np.maximum(gauss_max, weight * decay_factor * g)

        all_peak_values.append(gauss_max)

    # Max über alle Frequenzen
    result = np.maximum.reduce(all_peak_values)

    # Normierung auf output_max bei norm_x
    max_at_norm_x = np.max([
        (1 / (freq ** weight_exponent)) * np.exp(-decay * freq)
        for freq in range(1, max_freq + 1)
    ])
    return result / max_at_norm_x * output_max


def plot_proximity_max_freq_gaussian(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq_gaussian(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = Mittelwert
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit Gauß-Peaks (diskret, exp-decay)", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(x_vals, [mean_prox] * len(x_vals),
                 color='red', linestyle='--', linewidth=2,
                 label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit Gaussian-Peaks (diskret, exp-decay)")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, bins_hist, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")

    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=5, step=0.1, value=2, description="Sharpness Growth"),
    decay=FloatSlider(min=0, max=4, step=0.05, value=0.5, description="Decay"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=4, step=0.1, value=1, description="Weight Exponent"),
    freq_sharpness_exp=FloatSlider(min=0, max=4, step=0.1, value=1, description="Freq Sharpness Exp"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian(sharpness_base=10.0, sharpness_growth=2.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [ ]:
# Proximity Funktion mit Gauß-Peaks, decay & weight analog zu Cosinus-Version
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde

def proximity_max_freq_gaussian(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,              # Frequenz-Decay (wie bei Cosinus-Version, aber ohne x)
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_freq_values = []

    for freq in range(1, max_freq + 1):
        # Gewichtung pro Frequenz
        weight = 1 / (freq ** weight_exponent)
        decay_factor = 1 / (freq ** decay)

        # Frequenzabhängige Schärfe → steuert sigma
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)

        # sharpness hängt von x ab
        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        sigma = 1 / np.sqrt(sharpness + 1e-9)

        # Maxima der Gauss-Kette
        gauss_max = np.zeros_like(x, dtype=float)
        max_k = int(np.ceil(x.max() * freq)) + 1
        for k in range(1, max_k + 1):
            mu = k / freq
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)  # Höhe = 1
            gauss_max = np.maximum(gauss_max, g)

        # Anwenden von Gewichtung und decay
        val = weight * decay_factor * gauss_max
        all_freq_values.append(val)

    # Max über alle Frequenzen
    result = np.maximum.reduce(all_freq_values)

    # Normierung auf output_max bei norm_x
    max_at_norm_x = np.max([
        (1 / (freq ** weight_exponent)) * (1 / (freq ** decay))
        for freq in range(1, max_freq + 1)
    ])
    return result / max_at_norm_x * output_max


def plot_proximity_max_freq_gaussian(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq_gaussian(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = Mittelwert
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit Gauß-Peaks (freq-weighted)", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(x_vals, [mean_prox] * len(x_vals),
                 color='red', linestyle='--', linewidth=2,
                 label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit Gaussian-Peaks (freq & decay)")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, bins_hist, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")

    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=5, step=0.1, value=2, description="Sharpness Growth"),
    decay=FloatSlider(min=0, max=4, step=0.05, value=0.5, description="Decay"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=4, step=0.1, value=1, description="Weight Exponent"),
    freq_sharpness_exp=FloatSlider(min=0, max=4, step=0.1, value=1, description="Freq Sharpness Exp"),
)

interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian(sharpness_base=10.0, sharpness_growth=2.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [3]:
# Proximity Funktion mit Gauß-Peaks, fester Decay pro Kurve
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde


def proximity_max_freq_gaussian_decay(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,              # fester globaler Dämpfungsfaktor
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_curve_values = []

    for freq in range(1, max_freq + 1):
        # Frequenzgewichtung
        weight = 1 / (freq ** weight_exponent)

        # Frequenzabhängige Sharpness → steuert sigma
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)
        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        sigma = 1 / np.sqrt(sharpness + 1e-9)

        # feste Dämpfung pro Frequenz
        global_decay = np.exp(-decay * freq)

        # alle Gauss-Peaks für diese Frequenz
        gauss_max = np.zeros_like(x, dtype=float)
        max_k = int(np.ceil(x.max() * freq)) + 1
        for k in range(1, max_k + 1):
            mu = k / freq
            g = np.exp(-0.5 * ((x - mu) / sigma) ** 2)  # Höhe = 1
            gauss_max = np.maximum(gauss_max, g)

        # anwenden von Gewichtung + globalem decay
        val = weight * global_decay * gauss_max
        all_curve_values.append(val)

    # Maximum über alle Frequenzen
    result = np.maximum.reduce(all_curve_values)

    # Normierung auf output_max bei norm_x
    max_at_norm_x = np.max([
        (1 / (freq ** weight_exponent)) * np.exp(-decay * freq)
        for freq in range(1, max_freq + 1)
    ])
    return result / max_at_norm_x * output_max


def plot_proximity_max_freq_gaussian_decay(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq_gaussian_decay(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = Mittelwert
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Proximity-Funktion mit Gauß-Peaks (fester Decay)", fontsize=16, y=1.02)

    # Funktion plotten
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(x_vals, [mean_prox] * len(x_vals),
                 color='red', linestyle='--', linewidth=2,
                 label=f"∫mean ≈ {mean_prox:.3f}")
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit Gaussian-Peaks + festem Decay")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE
    counts, bins_hist, _ = axes[1].hist(
        y_vals, bins=50, range=(0, 1),
        alpha=0.4, color="skyblue", density=True, label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(x_dens, kde_vals * scaling_factor,
                 color="darkblue", lw=2, label="Dichte (KDE)")

    axes[1].axvline(mean_prox, color='red', linestyle='--',
                    linewidth=2, label=f"∫mean ≈ {mean_prox:.3f}")
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()


# Interaktive Kontrolle
interact(
    plot_proximity_max_freq_gaussian_decay,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=5, step=0.1, value=2, description="Sharpness Growth"),
    decay=FloatSlider(min=0, max=2, step=0.05, value=0.1, description="Decay"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=4, step=0.1, value=1, description="Weight Exponent"),
    freq_sharpness_exp=FloatSlider(min=0, max=4, step=0.1, value=1, description="Freq Sharpness Exp"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq_gaussian_decay(sharpness_base=10.0, sharpness_growth=2.0, decay=0.1, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>

In [2]:
# Proximity Funktion mit flexibler Anpassung
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
import numpy as np
from scipy.integrate import simpson
from scipy.stats import gaussian_kde

# Proximity-Funktion: bewertet Zahlenverhältnisse nach harmonischen Kriterien
def proximity_max_freq(
    x,
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.1,
    max_freq=4,
    weight_exponent=1.0,
    freq_sharpness_exp=1.0,
    norm_x=1.0,
    output_max=1.0
):
    x = np.array(x)
    all_freq_values = []

    for freq in range(1, max_freq + 1):
        weight = 1 / (freq ** weight_exponent)
        freq_sharpness_base = sharpness_base * (freq ** freq_sharpness_exp)
        freq_sharpness_growth = sharpness_growth * (freq ** freq_sharpness_exp)
        sharpness = freq_sharpness_base * (x ** freq_sharpness_growth)
        cos_term = ((1 + np.cos(2 * np.pi * freq * x)) / 2) ** sharpness

        # Normierung auf norm_x
        peak_sharpness = freq_sharpness_base * (norm_x ** freq_sharpness_growth)
        peak_value = ((1 + np.cos(2 * np.pi * freq * norm_x)) / 2) ** peak_sharpness
        cos_term_normalized = cos_term / peak_value

        val = weight * cos_term_normalized / (x ** decay)
        all_freq_values.append(val)

    result = np.maximum.reduce(all_freq_values)

    max_at_norm_x = np.max([
        (1 / (freq ** weight_exponent)) * 1.0 / (norm_x ** decay)
        for freq in range(1, max_freq + 1)
    ])
    return result / max_at_norm_x * output_max

# Plot-Funktion für Proximity und dessen Verteilung
def plot_proximity_max_freq(
    sharpness_base=10.0,
    sharpness_growth=2.0,
    decay=0.5,
    max_freq=6,
    weight_exponent=1.0,
    freq_sharpness_exp=0.5
):
    x_vals = np.linspace(1, 5, 1000)
    y_vals = proximity_max_freq(
        x_vals,
        sharpness_base=sharpness_base,
        sharpness_growth=sharpness_growth,
        decay=decay,
        max_freq=max_freq,
        weight_exponent=weight_exponent,
        freq_sharpness_exp=freq_sharpness_exp,
        output_max=1.0,
    )

    # Integral = numerischer Mittelwert der Funktion
    mean_prox = simpson(y_vals, x=x_vals) / (x_vals[-1] - x_vals[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Einfluss von Parametern auf Proximity-Funktion", fontsize=16, y=1.02)

    # Plot der Proximity-Funktion
    axes[0].plot(x_vals, y_vals, label=f"Proximity (∫mean ≈ {mean_prox:.3f})")
    axes[0].plot(
        x_vals,
        [mean_prox] * len(x_vals),
        color='red',
        linestyle='--',
        linewidth=2,
        label=f"∫mean ≈ {mean_prox:.3f}"
    )
    axes[0].set_xlabel("x (Verhältnis)")
    axes[0].set_ylabel("Proximity")
    axes[0].set_title("Proximity mit frequenzabhängiger Sharpness")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(True, linestyle="--", alpha=0.3)
    axes[0].legend()

    # Histogramm + KDE (Verteilung von y_vals)
    counts, bins_hist, _ = axes[1].hist(
        y_vals,
        bins=50,
        range=(0, 1),
        alpha=0.4,
        color="skyblue",
        density=True,
        label="Histogramm"
    )

    kde = gaussian_kde(y_vals)
    x_dens = np.linspace(0, 1, 5000)
    kde_vals = kde(x_dens)

    # Automatische Skalierung
    scaling_factor = max(counts) / max(kde_vals)
    axes[1].plot(
        x_dens,
        kde_vals * scaling_factor,
        color="darkblue",
        lw=2,
        label="Dichte (KDE)"
    )

    axes[1].axvline(
        mean_prox,
        color='red',
        linestyle='--',
        linewidth=2,
        label=f"∫mean ≈ {mean_prox:.3f}"
    )
    axes[1].set_title("Proximity-Verteilung")
    axes[1].set_xlabel("Proximity")
    axes[1].set_ylabel("Normierte Häufigkeit")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.subplots_adjust(top=0.90)
    plt.show()

# Interaktive Visualisierung
interact(
    plot_proximity_max_freq,
    sharpness_base=FloatSlider(min=1, max=50, step=1, value=10, description="Sharpness Base"),
    sharpness_growth=FloatSlider(min=0, max=5, step=0.1, value=2, description="Sharpness Growth"),
    decay=FloatSlider(min=0, max=4, step=0.05, value=0.1, description="Decay"),
    max_freq=IntSlider(min=1, max=6, step=1, value=4, description="Max Frequency"),
    weight_exponent=FloatSlider(min=0, max=4, step=0.1, value=1, description="Weight Exponent"),
    freq_sharpness_exp=FloatSlider(min=0, max=4, step=0.1, value=1, description="Freq Sharpness Exp"),
)


interactive(children=(FloatSlider(value=10.0, description='Sharpness Base', max=50.0, min=1.0, step=1.0), Floa…

<function __main__.plot_proximity_max_freq(sharpness_base=10.0, sharpness_growth=2.0, decay=0.5, max_freq=6, weight_exponent=1.0, freq_sharpness_exp=0.5)>